# Milestone 1: NLP Foundation & Semantic Similarity

- Perform Text cleaning, tokenization, and handling missing data.
- Generate embeddings using baseline models like TF-IDF and Word2Vec.
- Compute cosine similarity between a 'prompt' and 'options' and understand its concepts.
- Calculate mAP@3 and understand the concept of mAP.

In [7]:
import numpy as np, pandas as pd

In [8]:
df = pd.read_csv("../../data/train.csv")

In [9]:
df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [11]:
# Text cleaning, tokenization, handling missing data
import sys, os
sys.path.append(os.path.abspath("../.."))
from src.utils import OPTIONS, clean, tokenize, map_at_3, make_submission

df['q_clean'] = df['prompt'].map(clean)
for c in OPTIONS:
    df[c + '_clean'] = df[c].map(lambda x: clean(x, is_prompt=False))
df['q_tokens'] = df['prompt'].map(tokenize)

df[['prompt', 'q_clean', 'q_tokens']].head()

,prompt,q_clean,q_tokens
0,Pick the best possible answer: What is Martin ...,what is martin heidegger s view on the relatio...,"[what, is, martin, heidegger, s, view, on, the..."
1,What is accelerator-based light-ion fusion?,what is accelerator based light ion fusion,"[what, is, accelerator, based, light, ion, fus..."
2,Determine the correct option: What is the term...,what is the term used in astrophysics to descr...,"[what, is, the, term, used, in, astrophysics, ..."
3,Select the most accurate option: What is Marti...,what is martin heidegger s view on the relatio...,"[what, is, martin, heidegger, s, view, on, the..."
4,Identify the correct statement: What is the co...,what is the concept of simultaneity in einstei...,"[what, is, the, concept, of, simultaneity, in,..."


In [12]:
df.head()

,id,prompt,A,B,C,D,E,answer,q_clean,A_clean,B_clean,C_clean,D_clean,E_clean,q_tokens
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,what is martin heidegger s view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,"[what, is, martin, heidegger, s, view, on, the..."
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator based light ion fusion,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,accelerator based light ion fusion is a techni...,"[what, is, accelerator, based, light, ion, fus..."
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,what is the term used in astrophysics to descr...,blueshifting,redshifting,reddening,whitening,yellowing,"[what, is, the, term, used, in, astrophysics, ..."
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,what is martin heidegger s view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,"[what, is, martin, heidegger, s, view, on, the..."
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,what is the concept of simultaneity in einstei...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...,"[what, is, the, concept, of, simultaneity, in,..."


In [13]:
# Embeddings using TF-IDF and Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

corpus = pd.concat([df['q_clean']] + [df[c + '_clean'] for c in OPTIONS])
tfidf = TfidfVectorizer()
tfidf.fit(corpus)

sents = [s.split() for s in corpus]
w2v = Word2Vec(sents, vector_size=100, window=5, min_count=1, workers=1, seed=33)

def w2v_vec(text):
    words = [w for w in text.split() if w in w2v.wv]
    return np.mean([w2v.wv[w] for w in words], axis=0) if words else np.zeros(100)

In [14]:
# Cosine similarity between prompt and options
from sklearn.metrics.pairwise import cosine_similarity

def rank_options_tfidf(row):
    q = tfidf.transform([row['q_clean']])
    o = tfidf.transform([row[c + '_clean'] for c in OPTIONS])
    sims = cosine_similarity(q, o)[0]
    order = np.argsort(sims)[::-1]
    return [OPTIONS[i] for i in order]


def rank_options_w2v(row):
    q = w2v_vec(row['q_clean']).reshape(1, -1)
    o = np.vstack([w2v_vec(row[c + '_clean']) for c in OPTIONS])
    sims = cosine_similarity(q, o)[0]
    order = np.argsort(sims)[::-1]
    return [OPTIONS[i] for i in order]

In [15]:
# mAP@3
preds = df.apply(lambda r: rank_options_tfidf(r), axis=1)
print('mAP@3 tfidf:', round(map_at_3(df['answer'], preds), 4))

preds = df.apply(lambda r: rank_options_w2v(r), axis=1)
print('mAP@3 w2v:', round(map_at_3(df['answer'], preds), 4))

mAP@3 tfidf: 0.2971
mAP@3 w2v: 0.3465


In [16]:
test = pd.read_csv("../../data/test.csv")
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


In [17]:
test['q_clean'] = test['prompt'].map(clean)

for c in OPTIONS:
    test[c + '_clean'] = test[c].map(lambda x: clean(x, is_prompt=False))

preds = test.apply(lambda r: rank_options_w2v(r), axis=1)

sub = make_submission(test['id'], preds)
sub.to_csv('submission.csv', index=False)
sub.head()

,id,prediction
0,1,C A B
1,2,B E D
2,3,D C A
3,4,A C E
4,5,B D C
